# Stage 03 · Step 3 — Side-by-side comparison vs Stages 01 & 02

Reads `results/per_printer_tau_test*.csv` and `results/kpi_comparison.csv`, produces:
- Fleet annual cost / availability bar chart (3 stages).
- Cost vs availability scatter at the printer level (3 series).
- Per-printer τ heatmap for Stage 03 (15 printers × 6 components).
- Per-printer cost delta Stage 03 minus Stage 02 — the headline number.

In [ ]:
from __future__ import annotations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_models import PROJECT_ROOT
from sdg.schema import COMPONENT_IDS

RESULTS_DIR = PROJECT_ROOT / 'ml_models/03_rl+ssl/results'
comparison = pd.read_csv(RESULTS_DIR / 'kpi_comparison.csv')
comparison

In [ ]:
stage03 = pd.read_csv(RESULTS_DIR / 'per_printer_tau_test.csv')
stage_files = {
    'stage_01': RESULTS_DIR / 'per_printer_tau_test_stage01.csv',
    'stage_02': RESULTS_DIR / 'per_printer_tau_test_stage02.csv',
}
stage_dfs = {'stage_03': stage03}
for name, path in stage_files.items():
    if path.exists():
        stage_dfs[name] = pd.read_csv(path)
list(stage_dfs)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
stages_order = [s for s in ['stage_01', 'stage_02', 'stage_03'] if s in comparison['stage'].values]
ordered = comparison.set_index('stage').loc[stages_order]

axes[0].bar(stages_order, ordered['fleet_annual_cost'], color=['#888', '#3a85ff', '#ff7a3a'])
axes[0].set_ylabel('annual_cost (€/printer/yr)')
axes[0].set_title('Fleet annual cost on test printers')
for i, v in enumerate(ordered['fleet_annual_cost']):
    axes[0].text(i, v, f'{v:.2e}', ha='center', va='bottom')

axes[1].bar(stages_order, ordered['fleet_availability'], color=['#888', '#3a85ff', '#ff7a3a'])
axes[1].axhline(0.95, color='red', linestyle='--', alpha=0.5, label='95% threshold')
axes[1].set_ylabel('availability')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Fleet availability on test printers')
for i, v in enumerate(ordered['fleet_availability']):
    axes[1].text(i, v, f'{v:.3f}', ha='center', va='bottom')
axes[1].legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fleet_kpi_bars.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = {'stage_01': '#888', 'stage_02': '#3a85ff', 'stage_03': '#ff7a3a'}
for name, df in stage_dfs.items():
    ax.scatter(df['annual_cost'], df['availability'], label=name, color=colors.get(name, 'k'), alpha=0.7, s=60)
ax.axhline(0.95, color='red', linestyle='--', alpha=0.4, label='95% threshold')
ax.set_xscale('log')
ax.set_xlabel('per-printer annual_cost (€/yr)')
ax.set_ylabel('per-printer availability')
ax.set_title('Per-printer cost vs availability — three stages')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'per_printer_pareto.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
tau_cols = [f'tau_{c}' for c in COMPONENT_IDS]
tau_matrix = stage03.set_index('printer_id')[tau_cols].to_numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(
    np.log10(tau_matrix),
    aspect='auto',
    cmap='viridis',
)
ax.set_yticks(range(len(stage03)))
ax.set_yticklabels(stage03['printer_id'].astype(int).tolist())
ax.set_xticks(range(len(COMPONENT_IDS)))
ax.set_xticklabels(list(COMPONENT_IDS))
ax.set_xlabel('component')
ax.set_ylabel('printer_id')
ax.set_title('Stage 03 per-printer τ (log10 hours)')
plt.colorbar(im, ax=ax, label='log10(τ in hours)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'per_printer_tau_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

spread = pd.DataFrame({
    'tau_min': stage03[tau_cols].min().to_numpy(),
    'tau_median': stage03[tau_cols].median().to_numpy(),
    'tau_max': stage03[tau_cols].max().to_numpy(),
    'tau_max_over_min': stage03[tau_cols].max().to_numpy() / np.maximum(stage03[tau_cols].min().to_numpy(), 1e-9),
}, index=list(COMPONENT_IDS))
print('Per-component τ spread across test printers (max/min ratio):')
spread.round(2)

In [ ]:
if 'stage_02' in stage_dfs:
    s02 = stage_dfs['stage_02'].set_index('printer_id')[['annual_cost', 'availability']]
    s03 = stage_dfs['stage_03'].set_index('printer_id')[['annual_cost', 'availability']]
    delta = (s03 - s02).rename(columns={'annual_cost': 'cost_delta', 'availability': 'avail_delta'})
    delta['cost_delta_pct'] = 100.0 * delta['cost_delta'] / s02['annual_cost']

    fig, ax = plt.subplots(figsize=(9, 4))
    ordered_delta = delta['cost_delta_pct'].sort_values()
    bar_colors = ['seagreen' if v < 0 else 'salmon' for v in ordered_delta]
    ax.barh(ordered_delta.index.astype(str), ordered_delta.values, color=bar_colors)
    ax.axvline(0, color='black', linewidth=0.6)
    ax.set_xlabel('per-printer Stage 03 minus Stage 02 cost (% of Stage 02)')
    ax.set_ylabel('printer_id')
    ax.set_title('Stage 03 cost lift over Stage 02 (negative = better)')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'stage03_vs_stage02_delta.png', dpi=120, bbox_inches='tight')
    plt.show()

    summary = pd.Series({
        'printers_better': int((delta['cost_delta'] < 0).sum()),
        'printers_worse': int((delta['cost_delta'] > 0).sum()),
        'mean_cost_delta_pct': float(delta['cost_delta_pct'].mean()),
        'median_cost_delta_pct': float(delta['cost_delta_pct'].median()),
        'stage03_feasible': bool((stage_dfs['stage_03']['availability'] >= 0.95).all()),
    })
    print('Stage 03 vs Stage 02 head-to-head:')
    print(summary)
else:
    print('Stage 02 results not available — run 02_ssl/03_surrogate_search.ipynb first.')

## Headline

Read off `kpi_comparison.csv` for the report:
- `stage_03.fleet_value` is the headline scalar (lower is better).
- The ratio `stage_03.fleet_annual_cost / stage_02.fleet_annual_cost` is the relative improvement.
- The τ heatmap shows whether RL is actually using its per-printer freedom.